In [2]:
import os

In [3]:
def load_data(dir_location):
    file_list = os.listdir(dir_location)

    for i, file in enumerate(file_list):
        file_list[i] = os.path.join(dir_location, file)

    return file_list

In [ ]:
Voyager1_Files = os.path.join('SpaceY','RAG data','Voyager 1')
Voyager2_Files = os.path.join('SpaceY','RAG data','Voyager 2')

Voyager1_Data = load_data(Voyager1_Files)
Voyager2_Data = load_data(Voyager2_Files)

In [76]:
def File_open(files):

    documents = []
    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            documents.append(f.read())

    return documents

In [77]:
Voyager1 = File_open(Voyager1_Data)
Voyager2 = File_open(Voyager2_Data)

In [78]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

all_docs = []

for doc in Voyager1:
    all_docs.append({
        "mission": "Voyager1",
        "text": doc
    })

for doc in Voyager2:
    all_docs.append({
        "mission": "Voyager2",
        "text": doc
    })

print(all_docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[{'mission': 'Voyager1', 'text': "Entering Interstellar Space and Active Science Instruments\n\nVoyager 1 has traveled farther from Earth than any other human-made object. On February 17, 1998, at a distance of 69.4 astronomical units from the Sun, it overtook NASA's Pioneer 10 spacecraft to claim this record. As it continued its outward journey, the probe crossed the termination shock—the boundary where the solar wind abruptly slows down—on December 16, 2004, and entered the outer layer of the heliosphere known as the heliosheath. On August 25, 2012, Voyager 1 crossed the heliopause, leaving the Sun's magnetic influence behind and officially entering interstellar space.\n\nEven in this remote region, Voyager 1 continues to communicate with Earth via NASA's Deep Space Network. The spacecraft measures the properties of interstellar space using its remaining active instruments. These instruments include the Cosmic Ray Subsystem, which monitors high-energy galactic particles; the Low-Ener

In [79]:
print(len(Voyager1))
print(len(Voyager2))
print(len(all_docs))

20
20
40


In [80]:
query = "When was Voyager 1 launched?"
query_embedding = model.encode(
    query
)

In [81]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

scores= cosine_similarity(
    [query_embedding],
    [model.encode(doc["text"]) for doc in all_docs]
)

top_k_indices = np.argsort(scores[0])[-3:][::-1]

print(top_k_indices)
for idx in top_k_indices:
    print("=" * 50)
    print("Score:", scores[0, idx])
    print("Mission:", all_docs[idx]["mission"])
    print(all_docs[idx]["text"][:300])

[28 18  7]
Score: 0.7357788
Mission: Voyager2
Launch and Trajectory Details

Voyager 2 was launched on August 20, 1977, from Cape Canaveral, Florida, aboard a Titan IIIE-Centaur launch vehicle. Its twin, Voyager 1, was launched two weeks later on September 5, 1977. Despite its later launch, Voyager 1 reached Jupiter and Saturn sooner because it
Score: 0.70706487
Mission: Voyager1
What is Voyager 1?

Voyager 1 is a space probe launched by NASA on September 5, 1977, as part of the Voyager program. Its mission was designed to explore the outer solar system, specifically targeting Jupiter and Saturn. After completing its planetary flybys, Voyager 1 continued on a trajectory out 
Score: 0.6655704
Mission: Voyager1
In Depth: Voyager 1

Voyager 1 is a historic deep space mission managed by NASA's Jet Propulsion Laboratory (JPL) in Pasadena, California. The spacecraft represents a monumental achievement for the United States in the exploration of the outer planets. Designed to conduct close-up

In [82]:
context= ''

for idx in top_k_indices:
  context += all_docs[idx]['text']
  context += '\n\n'

print(context)

Launch and Trajectory Details

Voyager 2 was launched on August 20, 1977, from Cape Canaveral, Florida, aboard a Titan IIIE-Centaur launch vehicle. Its twin, Voyager 1, was launched two weeks later on September 5, 1977. Despite its later launch, Voyager 1 reached Jupiter and Saturn sooner because it was placed on a shorter, more direct trajectory.

Voyager 2 was launched on a longer, more circular orbit that kept its aphelion—its farthest point from the Sun—at 6.2 astronomical units, well short of Saturn's orbit, which allowed it to use a gravity assist at Jupiter to reach Saturn. In contrast, Voyager 1's initial orbit had an aphelion of 8.9 astronomical units, placing it closer to Saturn's orbit from the start. Voyager 1 overtook Voyager 2 in December 1977 as they traveled through the asteroid belt.


What is Voyager 1?

Voyager 1 is a space probe launched by NASA on September 5, 1977, as part of the Voyager program. Its mission was designed to explore the outer solar system, specific

In [84]:
prompt = f"""
You are a Space Mission Assistant.

Your task is to answer questions ONLY using the provided context.

Rules:
1. Use only information explicitly present in the context.
2. Do NOT use your own knowledge.
3. If the answer cannot be found in the context, reply exactly:
   "I could not find enough information in the provided documents."
4. Do not make assumptions or guesses.
5. If multiple context documents contain relevant information, combine them into a concise answer.
6. After the answer, provide the sources used.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
OPENROUTER_API = os.environ.get("OPENROUTER_API_KEY", "")

import requests
import json
from dotenv import load_dotenv

load_dotenv()
MODEL_API_KEY: str = os.getenv('MODEL_API_KEY')

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = requests.post(
  url=MODEL_API_KEY,
  headers={
    "Authorization": f"Bearer {OPENROUTER_API}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "poolside/laguna-xs.2:free",
    "messages": messages,
    "reasoning": {"enabled": True}
  })
)

if response.status_code == 200:
    response_data = response.json()
    if 'choices' in response_data and len(response_data['choices']) > 0:
        assistant_message = response_data['choices'][0]['message']
        final_answer = assistant_message.get('content')
        print(final_answer)
    else:
        print("Error: No choices found in the response.")
else:
    print(f"Error: OpenRouter API call failed with status code {response.status_code}")
    print(response.text)



Voyager 1 was launched on **September 5, 1977**.

**Sources used:**
- Launch and Trajectory Details
- Voyager 1
- In Depth: Voyager 1



In [89]:
import pickle

all_embeddings = []
for doc_item in all_docs:
    all_embeddings.append(model.encode(doc_item["text"]))

data = {
    "docs": all_docs,
    "embeddings": all_embeddings,
}

with open("RAG_model.pkl", "wb") as f:
    pickle.dump(data, f)